# 5d — Daily S2D Spatial Analysis
## JRA55\_FOSIRL vs Reanalysis: Rapid Daily Adjustment (Days 1–84)

This notebook orchestrates the `workflows/diagnostics/daily_drift/` workflow and provides
interactive exploration of rapid S2D adjustment through forecast day 84.

### Relationship to other workflows

| Notebook | Temporal Resolution | Scope |
|---|---|---|
| 5a\_refactor\_drift\_analysis | Monthly | Regional scalar indices (Niño3.4) |
| 5b\_refactor\_ic\_analysis | Start date | Coupled IC restart file audit |
| 5c\_refactor\_monthly\_spatial | Monthly | Full spatial monthly bias and drift |
| **5d (this notebook)** | Daily (days 1–84) | Rapid daily adjustment & weekly windows |

### Key design principles

1. Initialization date = lead day 0; first complete forecast day = lead day 1 ($d=1$).
2. Baseline day = 1 ($D(d=1) = 0$ by construction).
3. Window-aware gate completeness: `week_1` (1–7), `weeks_2_3` (8–21), `weeks_4_6` (22–42), `weeks_7_12` (43–84).
4. cftime noleap calendar retained during processing; leap days removed from obs.
5. Days 1–84 regional time series curves reveal when rapid adjustment occurs.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

import sys
import esp_lab

# Development-checkout fallback: a clean install exposes ``workflows`` directly.
REPO_ROOT = Path(esp_lab.__file__).resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workflows.diagnostics.daily_drift import bootstrap as bootstrap_workflow
from workflows.diagnostics.daily_drift import config as configuration
from workflows.diagnostics.daily_drift import diagnostics as diagnostics_workflow
from workflows.diagnostics.daily_drift import inventory as inventory_workflow
from workflows.diagnostics.daily_drift import plotting as plotting_workflow
from workflows.diagnostics.daily_drift import preprocess as preprocessing_workflow

from esp_lab.diagnostics.daily_core import (
    DailyDriftConfig, DailyGateStatus, DailyInventoryStatus,
    check_daily_lead_coverage, classify_daily_window_status,
    bootstrap_daily_spatial_ci, daily_significance_mask,
    daily_window_average, daily_regional_timeseries,
)
from esp_lab.utils.dask_utils import DaskConfig, get_cluster_client, close_cluster

DAILY_DIR = Path(configuration.__file__).resolve().parent
print('Daily S2D imports OK')


## Dask Setup

In [ ]:
from esp_lab.diagnostics.store import (
    diagnostic_store_is_valid, diagnostic_store_path,
    dataframe_fingerprint, file_fingerprint, load_diagnostic_store, save_diagnostic_store,
)

machine_env = os.environ.get('CLUSTER_TYPE', 'local')

dask_cfg = DaskConfig(
    cluster_type=machine_env,
    workers=8,
    cores=4,
    memory='16GB',
    walltime='04:00:00',
)

cluster, client = get_cluster_client(dask_cfg)
print(client)


## User Configuration

**All** parameters are defined here.  No other cell below needs changing.


In [ ]:
# ── Pilot toggle ────────────────────────────────────────────────
PILOT_ONLY    = True
PILOT_YEARS   = [1980, 1981, 1982]
PILOT_MONTHS  = [5]           # [5] = May only;  [5, 11] = both
VARIABLE      = 'TREFHT'      # Start with TREFHT; extend to PRECT etc.
GATE_STRICT   = False         # True = SystemExit(1) on gate failure
OBS_PATH      = None          # Set to ERA5_daily NetCDF path if available
N_BOOTSTRAP   = 1000
S2D_DIAG_ROOT = Path(os.environ.get('ESP_LAB_S2D_DIAG_ROOT', '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag'))
FIGURE_ROOT = Path(os.environ.get('ESP_LAB_FIGURE_OUTDIR', '/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag'))
OUTPUT_ROOT = S2D_DIAG_ROOT / 'multimodel' / 'leadtime_drift' / 'atm' / 'daily_spatial'
FIGURE_OUTDIR = FIGURE_ROOT / 'leadtime_drift' / 'atm' / 'daily_spatial'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)
REUSE_DIAGNOSTICS = True    # False forces raw-data recomputation

# ── Build DailyDriftConfig ───────────────────────────────────────
config = configuration.build_daily_config(
    pilot_only=PILOT_ONLY,
    pilot_years=PILOT_YEARS,
    pilot_months=PILOT_MONTHS,
    variable_filter=[VARIABLE],
    gate_strict=GATE_STRICT,
    n_bootstrap=N_BOOTSTRAP,
    output_root=OUTPUT_ROOT,
)

STORE_CONTEXT = {'observations': file_fingerprint(OBS_PATH)}
STORE_PATH = diagnostic_store_path(OUTPUT_ROOT, 'daily_drift', config, VARIABLE, context=STORE_CONTEXT)
STORE_VALID = REUSE_DIAGNOSTICS and diagnostic_store_is_valid(STORE_PATH, config, variable=VARIABLE, context=STORE_CONTEXT)
print('Diagnostic store:', STORE_PATH, '(reusable)' if STORE_VALID else '(will compute)')

print('Active years  :', config.active_years)
print('Active months :', config.active_months)
print('Members       :', config.members)
print('Lead days     :', f'1-{max(config.lead_days)}')
print('Window defs   :', list(config.window_defs))
print('Data dir      :', config.data_dir)


## Steps 3–5 — File Discovery, Inventory & Paired Readiness Gate

Probes daily postprocessed files (`.../ts/daily/0yr/`), calculates lead days
by date arithmetic, and classifies readiness across weekly windows.

**Outputs:**
- `${ESP_LAB_S2D_DIAG_ROOT}/multimodel/leadtime_drift/atm/daily_spatial/daily_inventory.csv`
- `${ESP_LAB_S2D_DIAG_ROOT}/multimodel/leadtime_drift/atm/daily_spatial/daily_missing_data_report.txt`
- `${ESP_LAB_S2D_DIAG_ROOT}/multimodel/leadtime_drift/atm/daily_spatial/daily_inventory_summary.json`


In [ ]:
%%time
inventory_df, report = inventory_workflow.run(
    pilot_only=PILOT_ONLY,
    variable=VARIABLE,
    season=None,
    skip_discovery=False,
    verbose=True,
    output_root=OUTPUT_ROOT,
    config=config,
)
STORE_CONTEXT['inventory'] = dataframe_fingerprint(inventory_df)
STORE_PATH = diagnostic_store_path(OUTPUT_ROOT, 'daily_drift', config, VARIABLE, context=STORE_CONTEXT)
STORE_VALID = REUSE_DIAGNOSTICS and diagnostic_store_is_valid(STORE_PATH, config, variable=VARIABLE, context=STORE_CONTEXT)
print('Post-inventory diagnostic store:', STORE_PATH, '(reusable)' if STORE_VALID else '(will compute)')
display(inventory_df.groupby(['experiment', 'daily_status'])
        .size().unstack(fill_value=0))


### Inspect daily paired readiness report

In [ ]:
print(report.format_report())

blocked = inventory_df[inventory_df['daily_status'] == 'BLOCKED']
if not blocked.empty:
    print(f'\n{len(blocked)} BLOCKED rows:')
    display(blocked[['experiment','init_date','member','missing_days','notes']]
            .head(20))
else:
    print('No BLOCKED rows — gate passes cleanly.')


## Steps 6–7 — Preprocess Approved Cohort

Opens approved daily files, retains cftime noleap calendar, converts units
(`TREFHT` → °C, `PRECT` → mm/day), and aligns forecast days 1–84.


In [ ]:
%%time
if STORE_VALID:
    results, boot_results = load_diagnostic_store(STORE_PATH)
    data, obs_da = {}, None
    print(f'Reused saved diagnostics: {STORE_PATH}')
else:
    data, obs_da = preprocessing_workflow.run_preprocess(
        config=config, inventory_df=inventory_df, report=report,
        field=VARIABLE, obs_path=OBS_PATH, verbose=True,
    )

for exp, months in data.items():
    for m, da in months.items():
        season_name = {5: 'May', 11: 'November'}.get(m, f'M{m:02d}')
        print(f'  {exp:<20s} {season_name}  shape={da.shape}  dims={da.dims}')


## Steps 8–9 — Spatial Daily Bias, Adjustment & Weekly Windows

Computes:
- $B_{m,s}(x,y,d) = F_{m,s}(x,y,d) - O_s(x,y,d)$
- $D_{m,s}(x,y,d) = B_{m,s}(x,y,d) - B_{m,s}(x,y,1)$  ($D(d=1) = 0$ by construction)
- $\Delta D_s(x,y,d) = D_{\text{test}}(x,y,d) - D_{\text{ref}}(x,y,d)$
- Weekly window averages: `week_1` (1–7), `weeks_2_3` (8–21), `weeks_4_6` (22–42), `weeks_7_12` (43–84)
- Days 1–84 daily regional time series curves


In [ ]:
%%time
if not STORE_VALID:
    results = diagnostics_workflow.run_diagnostics(
        data=data, config=config, obs_da=obs_da, verbose=True,
    )


### Days 1–84 regional adjustment time series

In [ ]:
import matplotlib.pyplot as plt

for init_month, month_res in results.items():
    season_name = {5: 'May', 11: 'November'}.get(init_month, f'M{init_month:02d}')
    ref_ts  = month_res.get('ref_ts_d')
    test_ts = month_res.get('test_ts_d')
    if ref_ts is not None and test_ts is not None:
        fig, ax = plt.subplots(figsize=(9, 4.5), dpi=120)
        ax.plot(ref_ts.d, ref_ts.values, label='D(Reanalysis)', color='crimson', lw=2)
        ax.plot(test_ts.d, test_ts.values, label='D(JRA55_FOSIRL)', color='navy', lw=2)
        ax.axhline(0, color='k', ls='--', lw=0.8)
        ax.set_xlabel('Forecast day', fontsize=11)
        ax.set_ylabel(f'Adjustment  ({config.variables[0].plot_units})', fontsize=11)
        ax.set_title(f'Days 1–84 adjustment evolution  |  {VARIABLE}  {season_name}', fontsize=12)
        ax.legend(fontsize=10)
        plt.tight_layout()
        plt.show()


## Step 10 — Bootstrap Confidence Intervals

Resamples paired initialization years with replacement ($N=1000$).
May and November starts are bootstrapped separately.


In [ ]:
%%time
if not STORE_VALID:
    boot_results = bootstrap_workflow.run_bootstrap(
        results=results, config=config, verbose=True,
    )
    save_diagnostic_store(STORE_PATH, results, boot_results, config=config,
                          variable=VARIABLE, workflow='daily_drift', context=STORE_CONTEXT)
    print(f'Saved diagnostics: {STORE_PATH}')


## Step 11 — Generate Figures and Tables

Saves figures to `${ESP_LAB_FIGURE_OUTDIR}/leadtime_drift/atm/daily_spatial/{season}/{window}/`:
- Adjustment maps (two-panel)
- Paired $\Delta D$ maps with significance stippling
- Days 1–84 regional time series plots
- Summary statistics tables


In [ ]:
%%time
summary_df = plotting_workflow.run_plotting(
    results=results,
    boot_results=boot_results,
    config=config,
    variable=VARIABLE,
    figure_outdir=FIGURE_OUTDIR,
    verbose=True,
)
display(summary_df)


## Extend to additional daily variables

> Run the cells below after verifying TREFHT looks physical.


In [ ]:
# Uncomment to run PRECT, LHFLX, SHFLX
# for var in ['PRECT', 'LHFLX', 'SHFLX']:
#     config_v = configuration.build_daily_config(pilot_only=PILOT_ONLY, variable_filter=[var])
#     data_v, obs_v = preprocessing_workflow.run_preprocess(config_v, inventory_df, report, field=var)
#     results_v     = diagnostics_workflow.run_diagnostics(data_v, config_v, obs_da=obs_v)
#     boot_v        = bootstrap_workflow.run_bootstrap(results_v, config_v)
#     plotting_workflow.run_plotting(results_v, boot_v, config_v, variable=var)


## Shutdown

In [ ]:
close_cluster(cluster, client)
